### Task 1: Download a public dataset and split into X and y

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Download and load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

df = pd.read_csv(url, sep=";")

# Display dataset
print("First 5 Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

# Split features and target
X = df.drop("quality", axis=1)
y = df["quality"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

print("\nFeatures:")
print(X.columns.tolist())

First 5 Rows:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2     

### Task 2: Fit Lasso Regression (L1)

In [5]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create Lasso model
lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.05, max_iter=10000)
)

# Train the model
lasso_model.fit(X_train, y_train)

# Get coefficients
lasso_coefficients = lasso_model.named_steps["lasso"].coef_

# Display coefficients
lasso_results = pd.DataFrame({
    "Feature": X.columns,
    "Lasso Coefficient": lasso_coefficients
})

print(lasso_results)

                 Feature  Lasso Coefficient
0          fixed acidity           0.007107
1       volatile acidity          -0.170689
2            citric acid           0.000000
3         residual sugar          -0.000000
4              chlorides          -0.010339
5    free sulfur dioxide          -0.000000
6   total sulfur dioxide          -0.036717
7                density          -0.000000
8                     pH          -0.000000
9              sulphates           0.077680
10               alcohol           0.277655


### Task 3: Fit Ridge Regression (L2) and compare coefficients

In [6]:
from sklearn.linear_model import Ridge

# Create Ridge model
ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

# Train the model
ridge_model.fit(X_train, y_train)

# Get Ridge coefficients
ridge_coefficients = ridge_model.named_steps["ridge"].coef_

# Compare Lasso and Ridge
comparison = pd.DataFrame({
    "Feature": X.columns,
    "Lasso Coefficient": lasso_coefficients,
    "Ridge Coefficient": ridge_coefficients
})

print(comparison.round(4))

                 Feature  Lasso Coefficient  Ridge Coefficient
0          fixed acidity             0.0071             0.0402
1       volatile acidity            -0.1707            -0.1792
2            citric acid             0.0000            -0.0272
3         residual sugar            -0.0000             0.0096
4              chlorides            -0.0103            -0.0890
5    free sulfur dioxide            -0.0000             0.0578
6   total sulfur dioxide            -0.0367            -0.1199
7                density            -0.0000            -0.0198
8                     pH            -0.0000            -0.0602
9              sulphates             0.0777             0.1468
10               alcohol             0.2777             0.2961


### Task 4: Fit ElasticNet and change l1_ratio

In [7]:
from sklearn.linear_model import ElasticNet

# Test different l1_ratio values
ratios = [0.2, 0.5, 0.8]

for ratio in ratios:

    # Create ElasticNet model
    elastic_model = make_pipeline(
        StandardScaler(),
        ElasticNet(
            alpha=0.05,
            l1_ratio=ratio,
            max_iter=10000
        )
    )

    # Train model
    elastic_model.fit(X_train, y_train)

    # Get coefficients
    coefficients = elastic_model.named_steps["elasticnet"].coef_

    print("\nL1 Ratio:", ratio)

    result = pd.DataFrame({
        "Feature": X.columns,
        "ElasticNet Coefficient": coefficients
    })

    print(result.round(4))


L1 Ratio: 0.2
                 Feature  ElasticNet Coefficient
0          fixed acidity                  0.0157
1       volatile acidity                 -0.1690
2            citric acid                  0.0000
3         residual sugar                  0.0000
4              chlorides                 -0.0739
5    free sulfur dioxide                  0.0257
6   total sulfur dioxide                 -0.0897
7                density                 -0.0015
8                     pH                 -0.0411
9              sulphates                  0.1274
10               alcohol                  0.2870

L1 Ratio: 0.5
                 Feature  ElasticNet Coefficient
0          fixed acidity                  0.0136
1       volatile acidity                 -0.1710
2            citric acid                  0.0000
3         residual sugar                  0.0000
4              chlorides                 -0.0502
5    free sulfur dioxide                  0.0000
6   total sulfur dioxide               

### Task 5: Select important restaurant features using Lasso

In [8]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# Create sample restaurant dataset
np.random.seed(42)

data = {
    "Average_Cost": np.random.randint(200, 1500, 200),
    "Delivery_Time": np.random.randint(15, 60, 200),
    "Review_Count": np.random.randint(10, 1000, 200),
    "Cuisine_Variety": np.random.randint(1, 15, 200),
    "Hygiene_Score": np.random.uniform(1, 10, 200),
    "Location_Score": np.random.uniform(1, 10, 200),
    "Ambience_Score": np.random.uniform(1, 10, 200),
    "Discount_Percentage": np.random.randint(0, 50, 200)
}

df = pd.DataFrame(data)

# Create restaurant rating
df["Rating"] = (
    1.5
    + 0.25 * df["Hygiene_Score"]
    + 0.20 * df["Location_Score"]
    + 0.15 * df["Ambience_Score"]
    - 0.015 * df["Delivery_Time"]
    + 0.0005 * df["Review_Count"]
    + np.random.normal(0, 0.3, 200)
)

# Keep ratings between 1 and 5
df["Rating"] = df["Rating"].clip(1, 5)

# Split features and target
X = df.drop("Rating", axis=1)
y = df["Rating"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create Lasso model
lasso_selector = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.05, max_iter=10000)
)

# Train model
lasso_selector.fit(X_train, y_train)

# Get coefficients
coefficients = lasso_selector.named_steps["lasso"].coef_

# Create feature selection table
feature_results = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients
})

# Select and drop features
selected_features = feature_results[
    feature_results["Coefficient"] != 0
]["Feature"].tolist()

dropped_features = feature_results[
    feature_results["Coefficient"] == 0
]["Feature"].tolist()

print("Feature Coefficients:")
print(feature_results.round(4))

print("\nSelected Features:")
print(selected_features)

print("\nDropped Features:")
print(dropped_features)

Feature Coefficients:
               Feature  Coefficient
0         Average_Cost       0.0000
1        Delivery_Time      -0.0788
2         Review_Count       0.0166
3      Cuisine_Variety      -0.0000
4        Hygiene_Score       0.3818
5       Location_Score       0.2329
6       Ambience_Score       0.1924
7  Discount_Percentage       0.0000

Selected Features:
['Delivery_Time', 'Review_Count', 'Hygiene_Score', 'Location_Score', 'Ambience_Score']

Dropped Features:
['Average_Cost', 'Cuisine_Variety', 'Discount_Percentage']
